In [ ]:
pip install pandas openpyxl pymupdf

In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from openai import OpenAI
from pathlib import Path

MODEL = "openai/gpt-5.2"          # OpenRouter model identifier
MAX_OUTPUT_TOKENS = 50000
TEMPERATURE = 0.1
TOP_P = 1

OUTPUT_CSV = "error_taxonomy.csv"
GENERATED_DIAGRAMS_DIR = Path("generated_diagrams")
REQUIREMENTS_DIR = Path("pure_requirements")
GOLD_STANDARD_DIR = Path("gold_standard")
TEMPLATE_FILE_NAME = 'template.xlsx'

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY environment variable not set.")

In [ ]:
# ============================================================
# Cell 2 : Configuration, Error Taxonomy & Prompt Settings
# ============================================================

from collections import OrderedDict

# ============================================================
# Prompt Type Mapping
# ============================================================

PROMPT_FILES = OrderedDict({
    "Simple_0": "Simple_0.puml",
    "Simple_1": "Simple_1.puml",
    "Simple_Few": "Simple_Few.puml",
    "CoT_0": "CoT_0.puml",
    "CoT_Few": "CoT_Few.puml"
})

# ============================================================
# Excel Sheet Mapping
# (must match template.xlsx exactly)
# ============================================================

SHEET_MAPPING = OrderedDict({
    "Simple_0": "Single Prompt_Zero Shot",
    "Simple_1": "Single Prompt_One Shot",
    "Simple_Few": "Single Prompt_Few Shot",
    "CoT_0": "CoT_Zero Shot",
    "CoT_Few": "CoT_Few Shot"
})

# ============================================================
# Error Taxonomy
# ============================================================

ERROR_TAXONOMY = OrderedDict({

    "Structural Errors": [

        "Missing Start Node",

        "Missing End Node",

        "Invalid Control Flow",

        "Incorrect Decision Structure",

        "Invalid Fork/Join",

        "PlantUML Syntax Errors"

    ],

    "Behavioural Errors": [

        "Missing Activity",

        "Hallucinated Activity",

        "Wrong Activity Sequence",

        "Missing Alternative Flow",

        "Missing Parallel Flow",

        "Incorrect Loop"

    ],

    "Requirement Coverage Errors": [

        "Unsupported Activity",

        "Missing Requirement Element",

        "Incomplete Activity",

        "Incorrect Requirement Mapping"

    ],

    "Actor & Swimlane Errors": [

        "Missing Actor",

        "Hallucinated Actor",

        "Wrong Actor Assignment",

        "Incorrect Swimlane Structure"

    ],

    "Semantic Interpretation Errors": [

        "Wrong Business Logic",

        "Wrong Decision Logic",

        "Ambiguous Interpretation",

        "Missing Exception Handling",

        "Incorrect Data Dependency"

    ],

    "Presentation Errors": [

        "Poor Activity Naming",

        "Overly Complex Layout",

        "Redundant Activities",

        "Inconsistent Granularity"

    ]

})

# ============================================================
# Flatten Error List
# ============================================================

ALL_ERRORS = []

for category in ERROR_TAXONOMY.values():
    ALL_ERRORS.extend(category)

print(f"Total Error Types : {len(ALL_ERRORS)}")

# ============================================================
# Empty Result Dictionary
# ============================================================

EMPTY_RESULT = OrderedDict()

for error in ALL_ERRORS:
    EMPTY_RESULT[error] = 0

# ============================================================
# LLM Configuration
# ============================================================

TEMPERATURE = 0.0

TOP_P = 0.1

MAX_RETRIES = 5

MAX_OUTPUT_TOKENS = 50000

JSON_REPAIR_RETRIES = 2

API_SLEEP = 2

# ============================================================
# Logging
# ============================================================

LOG_FILE = "evaluation_log.csv"

FAILED_JSON_FILE = "failed_json.txt"

CHECKPOINT_FILE = "taxonomy_checkpoint.pkl"

OUTPUT_EXCEL = "activity_diagram_error_taxonomy.xlsx"

# ============================================================
# Resume Support
# ============================================================

ENABLE_CHECKPOINT = True

SAVE_AFTER_EVERY_REQUIREMENT = True

# ============================================================
# Rule-based Error Detection
# ============================================================

RULE_BASED_ERRORS = {

    "Missing Start Node",

    "Missing End Node",

    "PlantUML Syntax Errors",

    "Invalid Fork/Join",

    "Incorrect Decision Structure",

    "Missing Activity",

    "Hallucinated Activity",

    "Wrong Activity Sequence",

    "Missing Parallel Flow",

    "Incorrect Swimlane Structure",

    "Missing Actor",

    "Hallucinated Actor",

    "Wrong Actor Assignment",

    "Redundant Activities"

}

# ============================================================
# LLM-evaluated Errors
# ============================================================

LLM_ERRORS = set(ALL_ERRORS) - RULE_BASED_ERRORS

# ============================================================
# JSON Schema Expected from LLM
# ============================================================

EXPECTED_JSON = OrderedDict()

for error in ALL_ERRORS:

    EXPECTED_JSON[error] = {
        "count": 0,
        "instances": []
    }

print("Configuration Loaded Successfully")

print("-"*60)

print("Prompt Types")

for p in PROMPT_FILES:
    print(" ", p)

print("-"*60)

print("Rule-based Errors :", len(RULE_BASED_ERRORS))

print("LLM Errors :", len(LLM_ERRORS))

print("Total Errors :", len(ALL_ERRORS))

In [ ]:
# ============================================================
# Cell 3 : Dataset Loading Utilities
# ============================================================

import os
import re
from pathlib import Path
from typing import Dict, List

import fitz  # PyMuPDF


# ============================================================
# Read Requirement PDF
# ============================================================

def extract_requirement_text(pdf_path: Path) -> str:
    """
    Extract complete text from a requirement PDF.
    """

    if not pdf_path.exists():
        raise FileNotFoundError(pdf_path)

    document = fitz.open(pdf_path)

    pages = []

    for page in document:
        pages.append(page.get_text())

    document.close()

    text = "\n".join(pages)

    return text.strip()


# ============================================================
# Read PlantUML File
# ============================================================

def read_plantuml(file_path: Path) -> str:
    """
    Reads a PlantUML (.puml/.txt) file.
    """

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()


# ============================================================
# Requirement ID Discovery
# ============================================================

def discover_requirement_ids() -> List[str]:
    """
    Discover requirement ids from the generated_diagrams folder.
    """

    reqs = []

    for item in GENERATED_DIAGRAMS_DIR.iterdir():

        if item.is_dir():

            reqs.append(item.name)

    reqs = sorted(reqs)

    return reqs


# ============================================================
# Paths
# ============================================================

def get_requirement_pdf(req_id: str) -> Path:

    return REQUIREMENTS_DIR / f"{req_id}.pdf"


def get_gold_diagram(req_id: str) -> Path:

    return GOLD_STANDARD_DIR / f"{req_id}.txt"


def get_generated_diagram(req_id: str,
                          prompt_name: str) -> Path:

    return GENERATED_DIAGRAMS_DIR / req_id / PROMPT_FILES[prompt_name]


# ============================================================
# Load One Requirement
# ============================================================

def load_requirement(req_id: str) -> Dict:

    requirement_text = extract_requirement_text(
        get_requirement_pdf(req_id)
    )

    gold_code = read_plantuml(
        get_gold_diagram(req_id)
    )

    diagrams = {}

    for prompt in PROMPT_FILES:

        diagrams[prompt] = read_plantuml(
            get_generated_diagram(req_id, prompt)
        )

    return {

        "requirement_id": req_id,

        "requirement": requirement_text,

        "gold": gold_code,

        "generated": diagrams

    }


# ============================================================
# Dataset Validation
# ============================================================

def validate_dataset():

    print("=" * 70)
    print("Validating Dataset...")
    print("=" * 70)

    requirement_ids = discover_requirement_ids()

    print(f"Requirements Found : {len(requirement_ids)}")

    missing_files = []

    for req in requirement_ids:

        # PDF

        pdf = get_requirement_pdf(req)

        if not pdf.exists():
            missing_files.append(str(pdf))

        # Gold Standard

        gold = get_gold_diagram(req)

        if not gold.exists():
            missing_files.append(str(gold))

        # Generated diagrams

        for prompt in PROMPT_FILES:

            generated = get_generated_diagram(req, prompt)

            if not generated.exists():

                missing_files.append(str(generated))

    if missing_files:

        print()

        print("Missing Files")

        print("-" * 70)

        for f in missing_files:
            print(f)

        raise RuntimeError(
            f"\nDataset validation failed.\n"
            f"Missing files : {len(missing_files)}"
        )

    print()

    print("Dataset Validation Successful")

    print(f"Requirements : {len(requirement_ids)}")

    print(f"Generated diagrams : {len(requirement_ids) * len(PROMPT_FILES)}")

    return requirement_ids


# ============================================================
# Preview Utility
# ============================================================

def preview_requirement(req_id: str):

    data = load_requirement(req_id)

    print("=" * 70)

    print("Requirement ID :", req_id)

    print("=" * 70)

    print()

    print("Requirement")

    print("-" * 70)

    print(data["requirement"][:800])

    print()

    print("Gold Diagram Length :", len(data["gold"]))

    print()

    for prompt in PROMPT_FILES:

        print(f"{prompt:<15} : {len(data['generated'][prompt])} characters")


# ============================================================
# Initialise Dataset
# ============================================================

REQUIREMENT_IDS = validate_dataset()

print()

print("Dataset Ready")

print(f"Total Requirements : {len(REQUIREMENT_IDS)}")

In [ ]:
# ============================================================
# Cell 4 : PlantUML Parser & Rule-Based Diagram Extraction
# ============================================================

import re
from collections import defaultdict


# ============================================================
# Regular Expressions
# ============================================================

ACTIVITY_PATTERN = re.compile(r':([^;]+);')
SWIMLANE_PATTERN = re.compile(r'^\s*\|(.+?)\|\s*$', re.MULTILINE)
ARROW_PATTERN = re.compile(r'(.+?)\s*-+.*?>\s*(.+)')
PARTITION_PATTERN = re.compile(r'partition\s+"?([^"]+)"?', re.IGNORECASE)
IF_PATTERN = re.compile(r'^\s*if\s*\(', re.IGNORECASE)
ELSE_PATTERN = re.compile(r'^\s*else', re.IGNORECASE)
ENDIF_PATTERN = re.compile(r'^\s*endif', re.IGNORECASE)
WHILE_PATTERN = re.compile(r'^\s*while\s*\(', re.IGNORECASE)
ENDWHILE_PATTERN = re.compile(r'^\s*endwhile', re.IGNORECASE)
FORK_PATTERN = re.compile(r'^\s*fork\b', re.IGNORECASE)
FORKAGAIN_PATTERN = re.compile(r'^\s*fork\s+again\b', re.IGNORECASE)
ENDFORK_PATTERN = re.compile(r'^\s*end\s+fork\b', re.IGNORECASE)


# ============================================================
# Clean Activity Name
# ============================================================

def clean_activity_name(name: str) -> str:

    name = name.strip()
    name = re.sub(r'<[^>]+>', '', name)
    name = re.sub(r'\[[^\]]+\]', '', name)
    name = re.sub(r'\s+', ' ', name)

    return name.strip()


# ============================================================
# Parse PlantUML
# ============================================================

def parse_plantuml(plantuml_code: str):

    diagram = {}

    lines = [
        line.strip()
        for line in plantuml_code.splitlines()
        if line.strip()
    ]

    diagram["lines"] = lines

    diagram["activities"] = []

    diagram["activity_counts"] = defaultdict(int)

    diagram["activity_set"] = set()

    diagram["actors"] = set()

    diagram["swimlanes"] = []

    diagram["decisions"] = []

    diagram["loops"] = []

    diagram["forks"] = []

    diagram["joins"] = []

    diagram["flows"] = []

    diagram["start_count"] = 0

    diagram["end_count"] = 0

    diagram["syntax_errors"] = []

    current_actor = None

    for line in lines:

        lower = line.lower()

        # ----------------------------------------------------
        # Start / End
        # ----------------------------------------------------

        if lower == "start":
            diagram["start_count"] += 1

        if lower == "end":
            diagram["end_count"] += 1

        # ----------------------------------------------------
        # Swimlane
        # ----------------------------------------------------

        lane = SWIMLANE_PATTERN.findall(line)

        if lane:

            current_actor = lane[0].strip()

            diagram["actors"].add(current_actor)

            diagram["swimlanes"].append(current_actor)

        # ----------------------------------------------------
        # Partition
        # ----------------------------------------------------

        part = PARTITION_PATTERN.findall(line)

        if part:

            current_actor = part[0].strip()

            diagram["actors"].add(current_actor)

            diagram["swimlanes"].append(current_actor)

        # ----------------------------------------------------
        # Activities
        # ----------------------------------------------------

        activities = ACTIVITY_PATTERN.findall(line)

        for activity in activities:

            activity = clean_activity_name(activity)

            diagram["activities"].append(activity)

            diagram["activity_set"].add(activity)

            diagram["activity_counts"][activity] += 1

        # ----------------------------------------------------
        # Decisions
        # ----------------------------------------------------

        if IF_PATTERN.search(line):

            diagram["decisions"].append(line)

        # ----------------------------------------------------
        # Loops
        # ----------------------------------------------------

        if WHILE_PATTERN.search(line):

            diagram["loops"].append(line)

        # ----------------------------------------------------
        # Fork / Join
        # ----------------------------------------------------

        if FORK_PATTERN.search(line):

            diagram["forks"].append(line)

        if FORKAGAIN_PATTERN.search(line):

            diagram["forks"].append(line)

        if ENDFORK_PATTERN.search(line):

            diagram["joins"].append(line)

        # ----------------------------------------------------
        # Control Flow
        # ----------------------------------------------------

        arrow = ARROW_PATTERN.findall(line)

        for src, dst in arrow:

            diagram["flows"].append(
                (
                    src.strip(),
                    dst.strip()
                )
            )

    return diagram


# ============================================================
# Structural Rule Checks
# ============================================================

def detect_missing_start(parsed):

    return max(0, 1 - parsed["start_count"])


def detect_missing_end(parsed):

    return max(0, 1 - parsed["end_count"])


def detect_fork_join_errors(parsed):

    return abs(
        len(parsed["forks"]) -
        len(parsed["joins"])
    )


def detect_decision_errors(parsed):

    if_count = sum(
        IF_PATTERN.search(line) is not None
        for line in parsed["lines"]
    )

    endif_count = sum(
        ENDIF_PATTERN.search(line) is not None
        for line in parsed["lines"]
    )

    return abs(if_count - endif_count)


def detect_loop_errors(parsed):

    while_count = sum(
        WHILE_PATTERN.search(line) is not None
        for line in parsed["lines"]
    )

    endwhile_count = sum(
        ENDWHILE_PATTERN.search(line) is not None
        for line in parsed["lines"]
    )

    return abs(while_count - endwhile_count)


# ============================================================
# Behaviour Comparison
# ============================================================

def compare_activity_sets(gold, generated):

    missing = gold["activity_set"] - generated["activity_set"]

    hallucinated = generated["activity_set"] - gold["activity_set"]

    return {

        "missing": sorted(missing),

        "hallucinated": sorted(hallucinated)

    }


def compare_swimlanes(gold, generated):

    missing = gold["actors"] - generated["actors"]

    hallucinated = generated["actors"] - gold["actors"]

    return {

        "missing": sorted(missing),

        "hallucinated": sorted(hallucinated)

    }


def compare_sequences(gold, generated):

    gold_seq = gold["activities"]

    gen_seq = generated["activities"]

    mismatch = 0

    n = min(len(gold_seq), len(gen_seq))

    for i in range(n):

        if gold_seq[i] != gen_seq[i]:

            mismatch += 1

    mismatch += abs(len(gold_seq) - len(gen_seq))

    return mismatch


# ============================================================
# Parse Gold + Generated
# ============================================================

def prepare_diagrams(gold_code, generated_code):

    gold = parse_plantuml(gold_code)

    generated = parse_plantuml(generated_code)

    return gold, generated

In [ ]:
# ============================================================
# Cell 5 : Rule-Based Error Detection & LLM Prompt Builder
# ============================================================

import json
import textwrap


# ============================================================
# Rule-based Error Detection
# ============================================================

def evaluate_rule_based_errors(gold_parsed, generated_parsed):

    result = EMPTY_RESULT.copy()

    # --------------------------------------------------------
    # Structural Errors
    # --------------------------------------------------------

    result["Missing Start Node"] = detect_missing_start(generated_parsed)

    result["Missing End Node"] = detect_missing_end(generated_parsed)

    result["Incorrect Decision Structure"] = detect_decision_errors(
        generated_parsed
    )

    result["Invalid Fork/Join"] = detect_fork_join_errors(
        generated_parsed
    )

    result["Incorrect Loop"] = detect_loop_errors(
        generated_parsed
    )

    # --------------------------------------------------------
    # Activity Comparison
    # --------------------------------------------------------

    comparison = compare_activity_sets(
        gold_parsed,
        generated_parsed
    )

    result["Missing Activity"] = len(comparison["missing"])

    result["Hallucinated Activity"] = len(comparison["hallucinated"])

    result["Wrong Activity Sequence"] = compare_sequences(
        gold_parsed,
        generated_parsed
    )

    # --------------------------------------------------------
    # Swimlane Comparison
    # --------------------------------------------------------

    lanes = compare_swimlanes(
        gold_parsed,
        generated_parsed
    )

    result["Missing Actor"] = len(lanes["missing"])

    result["Hallucinated Actor"] = len(lanes["hallucinated"])

    result["Wrong Actor Assignment"] = abs(
        len(gold_parsed["swimlanes"]) -
        len(generated_parsed["swimlanes"])
    )

    result["Incorrect Swimlane Structure"] = (
        result["Missing Actor"] +
        result["Hallucinated Actor"]
    )

    # --------------------------------------------------------
    # Parallel Flow
    # --------------------------------------------------------

    gold_parallel = len(gold_parsed["forks"])

    generated_parallel = len(generated_parsed["forks"])

    result["Missing Parallel Flow"] = max(
        0,
        gold_parallel - generated_parallel
    )

    # --------------------------------------------------------
    # Redundant Activities
    # --------------------------------------------------------

    redundant = 0

    for activity, count in generated_parsed["activity_counts"].items():

        if count > 1:

            redundant += count - 1

    result["Redundant Activities"] = redundant

    # --------------------------------------------------------
    # Syntax Errors
    # --------------------------------------------------------

    syntax = 0

    text = "\n".join(generated_parsed["lines"]).lower()

    if "@startuml" not in text:
        syntax += 1

    if "@enduml" not in text:
        syntax += 1

    result["PlantUML Syntax Errors"] = syntax

    # --------------------------------------------------------
    # Invalid Control Flow
    # --------------------------------------------------------

    if len(generated_parsed["flows"]) == 0:
        result["Invalid Control Flow"] = 1
    else:
        result["Invalid Control Flow"] = 0

    return result


# ============================================================
# LLM Prompt Builder
# ============================================================

def build_llm_prompt(requirement_text,
                     gold_diagram,
                     generated_diagram):

    prompt = f"""
You are an expert Software Engineering researcher.

Your task is to compare a generated UML Activity Diagram against:

1. Requirement Specification
2. Gold Standard UML Activity Diagram

Count ONLY semantic and requirement-related errors.

Do NOT count syntax errors or structural errors because they have already been evaluated separately.

Return ONLY valid JSON.

--------------------------------------------------------
Requirement
--------------------------------------------------------

{requirement_text}

--------------------------------------------------------
Gold Standard PlantUML
--------------------------------------------------------

{gold_diagram}

--------------------------------------------------------
Generated PlantUML
--------------------------------------------------------

{generated_diagram}

--------------------------------------------------------
Evaluate ONLY the following error taxonomy.
--------------------------------------------------------

Semantic Interpretation Errors

- Wrong Business Logic
- Wrong Decision Logic
- Ambiguous Interpretation
- Missing Exception Handling
- Incorrect Data Dependency

Requirement Coverage Errors

- Unsupported Activity
- Missing Requirement Element
- Incomplete Activity
- Incorrect Requirement Mapping

Behavioural Errors

- Missing Alternative Flow

Presentation Errors

- Poor Activity Naming
- Overly Complex Layout
- Inconsistent Granularity

--------------------------------------------------------
Output JSON Format
--------------------------------------------------------

{json.dumps(EXPECTED_JSON, indent=4)}

Rules:

1. Count every occurrence.

2. "count" must be an integer.

3. "instances" must be a list.

4. Never explain anything outside JSON.

5. Return only JSON.
"""

    return textwrap.dedent(prompt)

In [ ]:
# ============================================================
# Cell 6 : OpenRouter API, JSON Validation & LLM Evaluation
# ============================================================

import json
import time
import csv
from pathlib import Path


# ============================================================
# Logging
# ============================================================

def log_event(requirement_id,
              prompt_name,
              status,
              message="",
              tokens=0):

    file_exists = Path(LOG_FILE).exists()

    with open(LOG_FILE,
              "a",
              newline="",
              encoding="utf-8") as f:

        writer = csv.writer(f)

        if not file_exists:

            writer.writerow([
                "Requirement",
                "Prompt",
                "Status",
                "Tokens",
                "Message",
                "Timestamp"
            ])

        writer.writerow([
            requirement_id,
            prompt_name,
            status,
            tokens,
            message,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ])


# ============================================================
# JSON Extraction
# ============================================================

def extract_json(text):

    text = text.strip()

    if text.startswith("```json"):
        text = text[7:]

    if text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("JSON not found")

    return text[start:end + 1]


# ============================================================
# JSON Validation
# ============================================================

def validate_json(result):

    validated = {}

    for error in ALL_ERRORS:

        if error not in result:

            validated[error] = {
                "count": 0,
                "instances": []
            }

            continue

        item = result[error]

        if not isinstance(item, dict):

            validated[error] = {
                "count": 0,
                "instances": []
            }

            continue

        validated[error] = {

            "count": int(item.get("count", 0)),

            "instances": list(item.get("instances", []))

        }

    return validated


# ============================================================
# OpenRouter Call
# ============================================================

def call_llm(prompt):

    response = client.chat.completions.create(

        model=MODEL,

        temperature=TEMPERATURE,

        top_p=TOP_P,

        max_tokens=MAX_OUTPUT_TOKENS,

        messages=[
            {
                "role": "system",
                "content":
                "You are an expert in UML Activity Diagram verification."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    message = response.choices[0].message.content

    usage = getattr(response, "usage", None)

    tokens = 0

    if usage is not None:

        if hasattr(usage, "total_tokens"):

            tokens = usage.total_tokens

    return message, tokens


# ============================================================
# Retry Wrapper
# ============================================================

def evaluate_llm(requirement_id,
                 prompt_name,
                 prompt):

    last_exception = None

    for attempt in range(MAX_RETRIES):

        try:

            response_text, tokens = call_llm(prompt)

            json_text = extract_json(response_text)

            result = json.loads(json_text)

            result = validate_json(result)

            log_event(
                requirement_id,
                prompt_name,
                "SUCCESS",
                tokens=tokens
            )

            return result

        except Exception as ex:

            last_exception = ex

            log_event(
                requirement_id,
                prompt_name,
                "RETRY",
                str(ex)
            )

            time.sleep(API_SLEEP)

    log_event(
        requirement_id,
        prompt_name,
        "FAILED",
        str(last_exception)
    )

    with open(
        FAILED_JSON_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write("=" * 80 + "\n")
        f.write(requirement_id + "\n")
        f.write(prompt_name + "\n")
        f.write(str(last_exception) + "\n\n")

    empty = {}

    for error in LLM_ERRORS:

        empty[error] = {

            "count": 0,

            "instances": []

        }

    return empty


# ============================================================
# Merge Rule-Based + LLM Results
# ============================================================

def merge_results(rule_result,
                  llm_result):

    final = EMPTY_RESULT.copy()

    for error in ALL_ERRORS:

        if error in RULE_BASED_ERRORS:

            final[error] = rule_result.get(error, 0)

        else:

            value = llm_result.get(error, {})

            final[error] = int(
                value.get("count", 0)
            )

    return final

In [ ]:
# ============================================================
# Cell 7 : Evaluate One Diagram & One Requirement
# ============================================================

from copy import deepcopy


# ============================================================
# Evaluate a Single Diagram
# ============================================================

def evaluate_single_diagram(requirement_text,
                            gold_code,
                            generated_code,
                            requirement_id,
                            prompt_name):
    """
    Evaluate one generated activity diagram against
    the gold standard.
    """

    # --------------------------------------------------------
    # Parse diagrams
    # --------------------------------------------------------

    gold_parsed, generated_parsed = prepare_diagrams(
        gold_code,
        generated_code
    )

    # --------------------------------------------------------
    # Rule-based evaluation
    # --------------------------------------------------------

    rule_result = evaluate_rule_based_errors(
        gold_parsed,
        generated_parsed
    )

    # --------------------------------------------------------
    # LLM Evaluation
    # --------------------------------------------------------

    llm_prompt = build_llm_prompt(
        requirement_text,
        gold_code,
        generated_code
    )

    llm_result = evaluate_llm(
        requirement_id,
        prompt_name,
        llm_prompt
    )

    # --------------------------------------------------------
    # Merge
    # --------------------------------------------------------

    final_result = merge_results(
        rule_result,
        llm_result
    )

    return final_result


# ============================================================
# Evaluate One Requirement
# ============================================================

def evaluate_requirement(requirement_id):
    """
    Evaluate all five diagrams of one requirement.
    """

    data = load_requirement(requirement_id)

    results = {}

    for prompt_name in PROMPT_FILES:

        print(
            f"Evaluating {requirement_id} : {prompt_name}"
        )

        generated_code = data["generated"][prompt_name]

        result = evaluate_single_diagram(

            requirement_text=data["requirement"],

            gold_code=data["gold"],

            generated_code=generated_code,

            requirement_id=requirement_id,

            prompt_name=prompt_name
        )

        results[prompt_name] = result

    return results


# ============================================================
# Convert Result Dictionary to Excel Row
# ============================================================

def result_to_row(requirement_id,
                  result_dict):

    row = OrderedDict()

    row["Requirement ID"] = requirement_id

    for error in ALL_ERRORS:

        row[error] = int(
            result_dict.get(error, 0)
        )

    return row


# ============================================================
# Pretty Print
# ============================================================

def print_summary(requirement_id,
                  prompt_name,
                  result):

    print("=" * 70)

    print(requirement_id)

    print(prompt_name)

    print("-" * 70)

    total = 0

    for error in ALL_ERRORS:

        count = result.get(error, 0)

        if count > 0:

            total += count

            print(f"{error:<40} {count}")

    print("-" * 70)

    print("Total Errors :", total)

    print("=" * 70)


# ============================================================
# Test Utility
# ============================================================

def test_requirement(requirement_id):

    output = evaluate_requirement(requirement_id)

    for prompt_name, result in output.items():

        print_summary(
            requirement_id,
            prompt_name,
            result
        )

    return output

In [ ]:
# ============================================================
# Cell 8 : Evaluate Entire Dataset & Create DataFrames
# ============================================================

import os
import pickle
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# Checkpoint Utilities
# ============================================================

def load_checkpoint():

    if ENABLE_CHECKPOINT and os.path.exists(CHECKPOINT_FILE):

        with open(CHECKPOINT_FILE, "rb") as f:

            return pickle.load(f)

    checkpoint = {}

    for prompt in PROMPT_FILES:

        checkpoint[prompt] = []

    return checkpoint


def save_checkpoint(results):

    if not ENABLE_CHECKPOINT:
        return

    with open(CHECKPOINT_FILE, "wb") as f:

        pickle.dump(results, f)


# ============================================================
# Evaluate Entire Dataset
# ============================================================

def evaluate_dataset():

    all_results = load_checkpoint()

    completed = set()

    for prompt in PROMPT_FILES:

        for row in all_results[prompt]:

            completed.add(
                (
                    row["Requirement ID"],
                    prompt
                )
            )

    total = len(REQUIREMENT_IDS) * len(PROMPT_FILES)

    progress = tqdm(
        total=total,
        desc="Evaluating Diagrams"
    )

    progress.update(len(completed))

    for requirement_id in REQUIREMENT_IDS:

        try:

            data = load_requirement(requirement_id)

        except Exception as ex:

            print(f"Unable to load {requirement_id}")

            print(ex)

            continue

        for prompt_name in PROMPT_FILES:

            if (requirement_id, prompt_name) in completed:

                continue

            print(
                f"\n{requirement_id}  -->  {prompt_name}"
            )

            try:

                result = evaluate_single_diagram(

                    requirement_text=data["requirement"],

                    gold_code=data["gold"],

                    generated_code=data["generated"][prompt_name],

                    requirement_id=requirement_id,

                    prompt_name=prompt_name
                )

                row = result_to_row(
                    requirement_id,
                    result
                )

                all_results[prompt_name].append(row)

                log_event(
                    requirement_id,
                    prompt_name,
                    "COMPLETED"
                )

            except Exception as ex:

                print(ex)

                log_event(
                    requirement_id,
                    prompt_name,
                    "ERROR",
                    str(ex)
                )

            if SAVE_AFTER_EVERY_REQUIREMENT:

                save_checkpoint(all_results)

            progress.update(1)

    progress.close()

    return all_results


# ============================================================
# Convert Results to DataFrames
# ============================================================

def results_to_dataframes(results):

    dfs = {}

    for prompt in PROMPT_FILES:

        df = pd.DataFrame(results[prompt])

        if len(df):

            df = df.sort_values(
                "Requirement ID"
            ).reset_index(drop=True)

        dfs[prompt] = df

    return dfs


# ============================================================
# Summary Statistics
# ============================================================

def dataset_summary(dataframes):

    summary = {}

    for prompt in PROMPT_FILES:

        df = dataframes[prompt]

        if len(df) == 0:

            continue

        stats = {}

        for error in ALL_ERRORS:

            stats[error] = int(
                df[error].sum()
            )

        summary[prompt] = stats

    return summary


# ============================================================
# Run Evaluation
# ============================================================

results = evaluate_dataset()

dataframes = results_to_dataframes(results)

summary_statistics = dataset_summary(dataframes)

print("\nEvaluation Finished.")

for prompt in PROMPT_FILES:

    print(
        f"{prompt:<15} : {len(dataframes[prompt])} requirements"
    )

In [ ]:
# ============================================================
# Cell 9 : Populate Excel Template & Save Results
# ============================================================

from copy import copy
from openpyxl import load_workbook


# ============================================================
# Populate One Worksheet
# ============================================================

def populate_sheet(ws, df):
    """
    Writes dataframe values into the worksheet while preserving
    the formatting of the template.
    """

    # --------------------------------------------------------
    # Read Header Row
    # --------------------------------------------------------

    headers = {}

    for col in range(1, ws.max_column + 1):

        value = ws.cell(row=1, column=col).value

        if value is not None:

            headers[str(value).strip()] = col

    # --------------------------------------------------------
    # Clear Existing Data
    # --------------------------------------------------------

    if ws.max_row > 1:

        ws.delete_rows(2, ws.max_row)

    # --------------------------------------------------------
    # Write Data
    # --------------------------------------------------------

    for row_idx, (_, row) in enumerate(df.iterrows(), start=2):

        for column_name in df.columns:

            if column_name not in headers:
                continue

            col_idx = headers[column_name]

            ws.cell(
                row=row_idx,
                column=col_idx
            ).value = row[column_name]


# ============================================================
# Create Workbook
# ============================================================

def create_output_workbook():

    workbook = load_workbook(TEMPLATE_FILE_NAME)

    for prompt_name, sheet_name in SHEET_MAPPING.items():

        print(f"Writing {sheet_name}")

        worksheet = workbook[sheet_name]

        populate_sheet(
            worksheet,
            dataframes[prompt_name]
        )

    workbook.save(OUTPUT_EXCEL)

    print()

    print("=" * 60)

    print("Workbook Saved Successfully")

    print(OUTPUT_EXCEL)

    print("=" * 60)

    return workbook


# ============================================================
# Error Statistics
# ============================================================

def create_statistics_table():

    rows = []

    for prompt_name in PROMPT_FILES:

        df = dataframes[prompt_name]

        if len(df) == 0:
            continue

        total_errors = 0

        for error in ALL_ERRORS:

            total_errors += int(df[error].sum())

        rows.append({

            "Prompt": prompt_name,

            "Requirements": len(df),

            "Total Errors": total_errors,

            "Average Errors / Requirement":
                round(total_errors / len(df), 2)

        })

    return pd.DataFrame(rows)


# ============================================================
# Error Frequency Table
# ============================================================

def create_error_frequency_table():

    frequency = pd.DataFrame(index=ALL_ERRORS)

    for prompt_name in PROMPT_FILES:

        df = dataframes[prompt_name]

        frequency[prompt_name] = [

            int(df[e].sum()) if len(df) else 0

            for e in ALL_ERRORS
        ]

    frequency["Total"] = frequency.sum(axis=1)

    frequency = frequency.sort_values(
        "Total",
        ascending=False
    )

    return frequency


# ============================================================
# Save Workbook
# ============================================================

workbook = create_output_workbook()

statistics_df = create_statistics_table()

error_frequency_df = create_error_frequency_table()

print()

print(statistics_df)

print()

print(error_frequency_df.head(20))

In [ ]:
# ============================================================
# Cell 10 : Export Reports, Cleanup & Final Summary
# ============================================================

import shutil
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


# ============================================================
# Save CSV Files
# ============================================================

print("=" * 70)
print("Saving CSV files...")
print("=" * 70)

for prompt_name, df in dataframes.items():

    csv_path = RESULTS_DIR / f"{prompt_name}.csv"

    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8"
    )

    print(f"Saved : {csv_path.name}")


# ============================================================
# Save Statistics
# ============================================================

statistics_df.to_csv(
    RESULTS_DIR / "summary_statistics.csv",
    index=False
)

error_frequency_df.to_csv(
    RESULTS_DIR / "error_frequency.csv"
)


# ============================================================
# Export Statistics to Excel Workbook
# ============================================================

with pd.ExcelWriter(
    RESULTS_DIR / "analysis_reports.xlsx",
    engine="openpyxl"
) as writer:

    statistics_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    error_frequency_df.to_excel(
        writer,
        sheet_name="Error Frequency"
    )

    for prompt_name in PROMPT_FILES:

        dataframes[prompt_name].to_excel(
            writer,
            sheet_name=prompt_name,
            index=False
        )


# ============================================================
# Copy Final Workbook
# ============================================================

shutil.copy(
    OUTPUT_EXCEL,
    RESULTS_DIR / OUTPUT_EXCEL
)


# ============================================================
# Overall Statistics
# ============================================================

print()
print("=" * 70)
print("OVERALL SUMMARY")
print("=" * 70)

grand_total = 0

for prompt_name in PROMPT_FILES:

    df = dataframes[prompt_name]

    if len(df) == 0:
        continue

    prompt_total = int(
        df[ALL_ERRORS].sum().sum()
    )

    grand_total += prompt_total

    print(f"{prompt_name:<15} {prompt_total:>8} errors")

print("-" * 70)
print(f"Grand Total Errors : {grand_total}")
print()

print("Top 20 Most Frequent Errors")
print("-" * 70)

display(error_frequency_df.head(20))


# ============================================================
# Remove Checkpoint
# ============================================================

if ENABLE_CHECKPOINT and Path(CHECKPOINT_FILE).exists():

    Path(CHECKPOINT_FILE).unlink()

    print("\nCheckpoint removed.")


# ============================================================
# Final Status
# ============================================================

print()
print("=" * 70)
print("ERROR TAXONOMY GENERATION COMPLETED")
print("=" * 70)

print(f"Requirements Evaluated : {len(REQUIREMENT_IDS)}")
print(f"Prompt Types           : {len(PROMPT_FILES)}")
print(f"Total Diagrams         : {len(REQUIREMENT_IDS) * len(PROMPT_FILES)}")
print(f"Error Types            : {len(ALL_ERRORS)}")
print(f"Excel Output           : {OUTPUT_EXCEL}")
print(f"Results Folder         : {RESULTS_DIR.resolve()}")

print("\nGenerated Files:")

generated_files = [
    OUTPUT_EXCEL,
    "summary_statistics.csv",
    "error_frequency.csv",
    "analysis_reports.xlsx",
]

generated_files.extend(
    [f"{prompt}.csv" for prompt in PROMPT_FILES]
)

for file in generated_files:
    print(f"  ✓ {file}")

print("\nNotebook execution completed successfully.")